In [1]:
'''Auto resamples to 16kHz internally — your audio sample rate does not matter
Returns score between 1-5
output * 2 + 3 is the final scaling — model outputs -1 to 1 internally, scaled to MOS range
CPU supported — device="cpu" is the default'''

'Auto resamples to 16kHz internally — your audio sample rate does not matter\nReturns score between 1-5\noutput * 2 + 3 is the final scaling — model outputs -1 to 1 internally, scaled to MOS range\nCPU supported — device="cpu" is the default'

In [2]:
'''cd /Users/abey/Documents/UTMOS/model/simple
curl -L "https://huggingface.co/spaces/sarulab-speech/UTMOS-demo/resolve/main/epoch%3D3-step%3D7459.ckpt" -o "epoch=3-step=7459.ckpt"'''

'cd /Users/abey/Documents/UTMOS/model/simple\ncurl -L "https://huggingface.co/spaces/sarulab-speech/UTMOS-demo/resolve/main/epoch%3D3-step%3D7459.ckpt" -o "epoch=3-step=7459.ckpt"'

In [3]:
'''utmos) abey@PS017 simple % sed -i '' 's/weights_only=weights_only/weights_only=False/g' /Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/lightning_fabric/utilities/cloud_io.py
(utmos) abey@PS017 simple % grep "weights_only" /Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/lightning_fabric/utilities/cloud_io.py
    weights_only: Optional[bool] = None,
        weights_only: If ``True``, restricts loading to ``state_dicts`` of plain ``torch.Tensor`` and other primitive
            ``weights_only=False``. If loading checkpoint from an untrusted source, we recommend using
            ``weights_only=True``. For more information, please refer to the
            weights_only=False,
        if weights_only is None:
            weights_only = False
                f"Defaulting to `weights_only=False` for remote checkpoint: {path_or_url}."
                f" If loading a checkpoint from an untrustted source, we recommend using `weights_only=True`."
            weights_only=False,
            weights_only=False,
(utmos) abey@PS017 simple % cd /Users/abey/Documents/UTMOS/model/simple
curl -L "https://dl.fbaipublicfiles.com/fairseq/wav2vec/wav2vec_small.pt" -o "wav2vec_small.pt"
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  906M  100  906M    0     0  11.9M      0  0:01:16  0:01:16 --:--:-- 12.0M
(utmos) abey@PS017 simple % '''

'utmos) abey@PS017 simple % sed -i \'\' \'s/weights_only=weights_only/weights_only=False/g\' /Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/lightning_fabric/utilities/cloud_io.py\n(utmos) abey@PS017 simple % grep "weights_only" /Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/lightning_fabric/utilities/cloud_io.py\n    weights_only: Optional[bool] = None,\n        weights_only: If ``True``, restricts loading to ``state_dicts`` of plain ``torch.Tensor`` and other primitive\n            ``weights_only=False``. If loading checkpoint from an untrusted source, we recommend using\n            ``weights_only=True``. For more information, please refer to the\n            weights_only=False,\n        if weights_only is None:\n            weights_only = False\n                f"Defaulting to `weights_only=False` for remote checkpoint: {path_or_url}."\n                f" If loading a checkpoint from an untrustted source, we recommend using `weights_only=True`."\n      

In [4]:
!conda activate utmos
!pip install ipykernel --upgrade
!python -m ipykernel install --user --name utmos --display-name "Python (utmos)"


CondaError: Run 'conda init' before 'conda activate'

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Installed kernelspec utmos in /Users/abey/Library/Jupyter/kernels/utmos


In [5]:
# ============================================================
# CELL 1 — Setup
# ============================================================
import sys
import os

UTMOS_DIR = "/Users/abey/Documents/UTMOS/model/simple"
os.chdir(UTMOS_DIR)
if UTMOS_DIR not in sys.path:
    sys.path.insert(0, UTMOS_DIR)

print(f"Working directory: {os.getcwd()}")
print("✅ Setup done")

Working directory: /Users/abey/Documents/UTMOS/model/simple
✅ Setup done


In [6]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import torch
import torchaudio
import pandas as pd
from score import Score

print("✅ Imports done")

INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX


✅ Imports done


In [7]:
# ============================================================
# CELL 3 — Load model and set threshold
# ============================================================
CKPT_PATH       = "/Users/abey/Documents/UTMOS/model/simple/epoch=3-step=7459.ckpt"
UTMOS_THRESHOLD = 3.0  # calibrate with editor later

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

scorer = Score(
    ckpt_path=CKPT_PATH,
    input_sample_rate=16000,
    device="cpu"
)

print(f"✅ UTMOS model loaded")
print(f"Threshold: {UTMOS_THRESHOLD}")

DEBUG:fsspec.local:open file: /Users/abey/Documents/UTMOS/model/simple/epoch=3-step=7459.ckpt


Using device: cpu


Lightning automatically upgraded your loaded checkpoint from v1.5.9 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint epoch=3-step=7459.ckpt`
DEBUG:hydra.core.utils:Setting JobRuntime:name=UNKNOWN_NAME
DEBUG:hydra.core.utils:Setting JobRuntime:name=utils
DEBUG:hydra.core.utils:Setting JobRuntime:name=utils
/Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ UTMOS model loaded
Threshold: 3.0


In [8]:
# ============================================================
# CELL 4 — score_single_file function
# ============================================================
def utmos_score(audio_path):
    wav, sr = torchaudio.load(audio_path)
    
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    
    scorer.in_sr = sr
    scorer.resampler = torchaudio.transforms.Resample(
        orig_freq=sr,
        new_freq=16000,
        resampling_method="sinc_interpolation",
        lowpass_filter_width=6,
        dtype=torch.float32,
    )
    
    score = scorer.score(wav)
    return round(float(score[0]), 3)

print("✅ utmos_score defined")

✅ utmos_score defined


In [9]:
# ============================================================
# CELL 5 — Configure paths and validate structure
# ============================================================
BASE_DIR   = "/Users/abey/Documents/UTMOS"
MODELS_DIR = os.path.join(BASE_DIR, "models")

# ── check models folder ──
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

# ── discover model folders ──
model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

# ── discover wav files per model ──
model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([
        f for f in os.listdir(model_path)
        if f.endswith(".wav")
    ])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

# ── validate all models have identical filenames ──
reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
total        = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")

✅ Models found: ['m1', 'm2']
   m1: 2 samples
   m2: 2 samples
✅ All models have identical filenames

Ready: 2 models × 2 samples = 4 evaluations


In [10]:
# ============================================================
# CELL 6 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        audio_path  = os.path.join(MODELS_DIR, model, wav_file)

        print(f"\n  Sample : {sample_name}")

        score     = utmos_score(audio_path)
        passed    = score >= UTMOS_THRESHOLD
        
        print(f"  UTMOS  : {score} → {'✅ PASS' if passed else '❌ FAIL'}")

        results.append({
            "Model"   : model,
            "Sample"  : sample_name,
            "UTMOS"   : score,
            "Pass"    : "✅" if passed else "❌",
        })

print("\n\nAll evaluations complete.")


Model: m1

  Sample : mac_tts_test


/Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/var/folders/f1/4wcysxy93zg1k5d8bhzshxc80000gq/T/ipykernel_12380/1925867059.py:11: UserWarning: "sinc_interpolation" resampling method name is being deprecated and replaced by "sinc_interp_hann" in the next release. The default behavior remains unchanged.
  scorer.resampler = torchaudio.transforms.Resample(


  UTMOS  : 4.022 → ✅ PASS

  Sample : robotic_test
  UTMOS  : 2.074 → ❌ FAIL

Model: m2

  Sample : mac_tts_test
  UTMOS  : 4.022 → ✅ PASS

  Sample : robotic_test
  UTMOS  : 2.074 → ❌ FAIL


All evaluations complete.


In [11]:
# ============================================================
# CELL 7 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[["Model", "Sample", "UTMOS", "Pass"]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    utmos_vals  = model_df["UTMOS"]
    pass_count  = (model_df["Pass"] == "✅").sum()
    total       = len(model_df)

    summary_rows.append({
        "Model"       : model,
        "Segments"    : total,
        "Mean UTMOS"  : round(utmos_vals.mean(), 3),
        "Median UTMOS": round(utmos_vals.median(), 3),
        "Min UTMOS"   : round(utmos_vals.min(), 3),
        "Max UTMOS"   : round(utmos_vals.max(), 3),
        "Pass Rate"   : f"{pass_count}/{total}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Pass Rate (highest first)")
print("Tiebreak1 → Median UTMOS (highest first)")
print("Tiebreak2 → Min UTMOS (highest first — best worst case)\n")

# extract numeric pass rate for sorting
summary_df["_pass_num"] = summary_df["Pass Rate"].apply(
    lambda x: int(x.split("/")[0])
)

ranking = summary_df.sort_values(
    by=[
        "_pass_num",    # Pass Rate — highest first
        "Median UTMOS", # Median — highest first
        "Min UTMOS",    # Min — highest first (best worst case)
    ],
    ascending=[
        False,  # Pass Rate — higher is better
        False,  # Median UTMOS — higher is better
        False,  # Min UTMOS — higher is better
    ]
)[[
    "Model", "Pass Rate", "Median UTMOS", "Min UTMOS", "Mean UTMOS"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Pass Rate    → % of segments production ready — primary decision metric")
print("Median UTMOS → typical naturalness score — tiebreaker")
print("Min UTMOS    → worst segment — how bad do failures get")
print("             → large gap between Median and Min = unpredictable failures")
print("Threshold    → 3.0 is starting point, calibrate with editor")



========== FULL PER-SEGMENT RESULTS ==========
Model       Sample  UTMOS Pass
   m1 mac_tts_test  4.022    ✅
   m1 robotic_test  2.074    ❌
   m2 mac_tts_test  4.022    ✅
   m2 robotic_test  2.074    ❌

========== MODEL COMPARISON SUMMARY ==========
Model  Segments  Mean UTMOS  Median UTMOS  Min UTMOS  Max UTMOS Pass Rate
   m1         2       3.048         3.048      2.074      4.022       1/2
   m2         2       3.048         3.048      2.074      4.022       1/2

========== MODEL RANKING ==========
Primary   → Pass Rate (highest first)
Tiebreak1 → Median UTMOS (highest first)
Tiebreak2 → Min UTMOS (highest first — best worst case)

Model Pass Rate  Median UTMOS  Min UTMOS  Mean UTMOS
   m1       1/2         3.048      2.074       3.048
   m2       1/2         3.048      2.074       3.048

========== WHAT TO LOOK FOR ==========
Pass Rate    → % of segments production ready — primary decision metric
Median UTMOS → typical naturalness score — tiebreaker
Min UTMOS    → worst segment 

In [12]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
